# HW5: Hierarchical Clustering, DBSCAN, and Cluster Evaluation

In this lab, we compare different clustering methods and evaluate them.

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import json
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_circles
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    fowlkes_mallows_score,
    rand_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture

INITIALS = "senthilnathan_t"
FIG_DIR = Path("figs_results")
FIG_DIR.mkdir(exist_ok=True)

def figure_path(task_num, fig_num):
    return FIG_DIR / f"a5_task{task_num}_fig{fig_num}_{INITIALS}.png"

def save_figure(fig, task_num, fig_num):
    path = figure_path(task_num, fig_num)
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return str(path)

def relabel_for_plot(labels):
    unique = [label for label in sorted(set(labels)) if label != -1]
    mapping = {label: idx for idx, label in enumerate(unique)}
    return np.array([mapping.get(label, -1) for label in labels])

def cluster_counts(labels):
    counts = Counter(int(label) for label in labels)
    ordered = {}
    for label in sorted(counts):
        key = "noise" if label == -1 else f"cluster_{label}"
        ordered[key] = int(counts[label])
    return ordered

def cluster_text(counts):
    parts = []
    for key, value in counts.items():
        if key.startswith("cluster_"):
            idx = int(key.split("_")[1]) + 1
            parts.append(f"cluster {idx}={value}")
        else:
            parts.append(f"{key}={value}")
    return ", ".join(parts)

def cut_height_for_two_clusters(Z):
    return float((Z[-1, 2] + Z[-2, 2]) / 2.0)

def plot_clusters(ax, X, labels, title):
    mapped = relabel_for_plot(labels)
    unique_clusters = [label for label in sorted(set(mapped)) if label != -1]
    colors = plt.cm.tab10(np.linspace(0, 1, max(1, len(unique_clusters))))
    for idx, label in enumerate(unique_clusters):
        mask = mapped == label
        ax.scatter(X[mask, 0], X[mask, 1], s=18, color=colors[idx], alpha=0.85, label=f"Cluster {label + 1}")
    if np.any(mapped == -1):
        mask = mapped == -1
        ax.scatter(X[mask, 0], X[mask, 1], s=22, color="#333333", marker="x", label="Noise")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")

def plot_dbscan_types(ax, X, labels, core_mask, title):
    mapped = relabel_for_plot(labels)
    unique_clusters = [label for label in sorted(set(mapped)) if label != -1]
    colors = plt.cm.tab10(np.linspace(0, 1, max(1, len(unique_clusters))))
    for idx, label in enumerate(unique_clusters):
        mask = mapped == label
        core_points = mask & core_mask
        border_points = mask & (~core_mask)
        ax.scatter(X[core_points, 0], X[core_points, 1], s=36, color=colors[idx], alpha=0.9)
        ax.scatter(X[border_points, 0], X[border_points, 1], s=18, color=colors[idx], edgecolors="black", linewidths=0.4, alpha=0.9)
    noise_points = labels == -1
    if np.any(noise_points):
        ax.scatter(X[noise_points, 0], X[noise_points, 1], s=24, color="#333333", marker="x")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor="#666666", markeredgecolor="#666666", markersize=7, label="Core"),
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor="#bbbbbb", markeredgecolor="black", markersize=6, label="Border"),
        Line2D([0], [0], marker="x", linestyle="", color="#333333", markersize=7, label="Noise"),
    ]
    ax.legend(handles=legend_handles, loc="best", fontsize=8, frameon=False)

def safe_silhouette(X, labels):
    labels = np.array(labels)
    mask = labels != -1
    if mask.sum() < 2:
        return None
    filtered = labels[mask]
    if len(set(filtered)) < 2:
        return None
    return float(silhouette_score(X[mask], filtered))

--------------------------------------------------
## Question 1: Hierarchical clustering on well-separated clusters
--------------------------------------------------

In [2]:
np.random.seed(0)
C1, _ = make_blobs(n_samples=200, centers=[[0, 0]], cluster_std=0.5)
C2, _ = make_blobs(n_samples=200, centers=[[4, 4]], cluster_std=0.5)
X = np.vstack([C1, C2])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(X[:, 0], X[:, 1], s=16, color="#2a6f97", alpha=0.8)
ax.set_title("Task 1 raw data")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
task1_fig1 = save_figure(fig, 1, 1)

task1_labels = {}
task1_heights = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, method in zip(axes, ["single", "complete", "average"]):
    Z = linkage(X, method=method, optimal_ordering=True)
    labels = fcluster(Z, t=2, criterion="maxclust") - 1
    task1_labels[method] = labels
    task1_heights[method] = cut_height_for_two_clusters(Z)
    plot_clusters(ax, X, labels, f"{method.title()} linkage")
task1_fig2 = save_figure(fig, 1, 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, method in zip(axes, ["single", "complete", "average"]):
    Z = linkage(X, method=method, optimal_ordering=True)
    dendrogram(Z, no_labels=True, color_threshold=task1_heights[method], ax=ax)
    ax.axhline(task1_heights[method], color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{method.title()} dendrogram")
    ax.set_xlabel("Observations")
    ax.set_ylabel("Merge height")
task1_fig3 = save_figure(fig, 1, 3)

task1_results = {
    "figures": {"raw": task1_fig1, "assignments": task1_fig2, "dendrograms": task1_fig3},
    "cluster_sizes": {method: cluster_counts(labels) for method, labels in task1_labels.items()},
    "pairwise_rand": {
        "single_vs_complete": rand_score(task1_labels["single"], task1_labels["complete"]),
        "single_vs_average": rand_score(task1_labels["single"], task1_labels["average"]),
        "complete_vs_average": rand_score(task1_labels["complete"], task1_labels["average"]),
    }
}

print("Task 1 cluster sizes")
print(json.dumps(task1_results["cluster_sizes"], indent=2))
print("\nPairwise Rand agreement")
for name, value in task1_results["pairwise_rand"].items():
    print(f"{name}: {value:.3f}")

Task 1 cluster sizes
{
  "single": {
    "cluster_0": 200,
    "cluster_1": 200
  },
  "complete": {
    "cluster_0": 200,
    "cluster_1": 200
  },
  "average": {
    "cluster_0": 200,
    "cluster_1": 200
  }
}

Pairwise Rand agreement
single_vs_complete: 1.000
single_vs_average: 1.000
complete_vs_average: 1.000


![Task 1 Raw Data](figs_results/a5_task1_fig1_senthilnathan_t.png)

![Task 1 assignments](figs_results/a5_task1_fig2_senthilnathan_t.png)

![Task 1 dendrograms](figs_results/a5_task1_fig3_senthilnathan_t.png)

All three linkage rules gave the same final split at `K=2`, with cluster 1=200, cluster 2=200. The Rand agreement between each pair of methods was 1.000, so this is a case where the clusters are separated enough that the linkage choice does not change the answer.

--------------------------------------------------
## Question 2: Chaining effect
--------------------------------------------------

In [3]:
np.random.seed(0)
A, _ = make_blobs(n_samples=150, centers=[[0, 0]], cluster_std=0.5)
B, _ = make_blobs(n_samples=150, centers=[[6, 0]], cluster_std=0.5)
chain = np.column_stack([np.linspace(0, 6, 50), np.random.normal(0, 0.1, 50)])
X_chain = np.vstack([A, B, chain])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(X_chain[:, 0], X_chain[:, 1], s=16, color="#2a6f97", alpha=0.8)
ax.set_title("Task 2 raw data with bridge")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
task2_fig1 = save_figure(fig, 2, 1)

single_labels = fcluster(linkage(X_chain, method="single", optimal_ordering=True), t=2, criterion="maxclust") - 1
complete_labels = fcluster(linkage(X_chain, method="complete", optimal_ordering=True), t=2, criterion="maxclust") - 1
db = DBSCAN(eps=0.35, min_samples=5).fit(X_chain)
db_labels = db.labels_

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
plot_clusters(axes[0], X_chain, single_labels, "Single linkage")
plot_clusters(axes[1], X_chain, complete_labels, "Complete linkage")
plot_clusters(axes[2], X_chain, db_labels, "DBSCAN eps=0.35")
task2_fig2 = save_figure(fig, 2, 2)

task2_results = {
    "figures": {"raw": task2_fig1, "comparison": task2_fig2},
    "single_cluster_sizes": cluster_counts(single_labels),
    "complete_cluster_sizes": cluster_counts(complete_labels),
    "dbscan_cluster_sizes": cluster_counts(db_labels),
}

print("Single linkage sizes:", task2_results["single_cluster_sizes"])
print("Complete linkage sizes:", task2_results["complete_cluster_sizes"])
print("DBSCAN sizes:", task2_results["dbscan_cluster_sizes"])

Single linkage sizes: {'cluster_0': 349, 'cluster_1': 1}
Complete linkage sizes: {'cluster_0': 174, 'cluster_1': 176}
DBSCAN sizes: {'noise': 14, 'cluster_0': 179, 'cluster_1': 157}


![Task 2 Raw Data with bridge](figs_results/a5_task2_fig1_senthilnathan_t.png)

![Task 2 comparison](figs_results/a5_task2_fig2_senthilnathan_t.png)

Single linkage almost merged the full dataset into one group: cluster 1=349, cluster 2=1. Complete linkage stayed balanced with cluster 1=174, cluster 2=176. DBSCAN produced noise=14, cluster 1=179, cluster 2=157. The bridge points create a short-step path, which is enough for single linkage but not enough for complete linkage.

--------------------------------------------------
## Question 3: DBSCAN and density differences
--------------------------------------------------

In [4]:
np.random.seed(0)
C1, _ = make_blobs(n_samples=300, centers=[[0, 0]], cluster_std=0.3)
C2, _ = make_blobs(n_samples=300, centers=[[5, 5]], cluster_std=1.2)
X_db = np.vstack([C1, C2])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(X_db[:, 0], X_db[:, 1], s=16, color="#2a6f97", alpha=0.8)
ax.set_title("Task 3 raw data")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
task3_fig1 = save_figure(fig, 3, 1)

task3_results = {"figures": {"raw": task3_fig1}, "eps_results": {}}
eps_values = [0.2, 0.5, 1.0]
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5))
for ax, eps in zip(axes, eps_values):
    db = DBSCAN(eps=eps).fit(X_db)
    labels = db.labels_
    core_mask = np.zeros(len(X_db), dtype=bool)
    core_mask[db.core_sample_indices_] = True
    border_mask = (labels != -1) & (~core_mask)
    noise_mask = labels == -1
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    plot_dbscan_types(ax, X_db, labels, core_mask, f"eps={eps}\nclusters={n_clusters}, noise={int(noise_mask.sum())}")
    task3_results["eps_results"][str(eps)] = {
        "n_clusters": int(n_clusters),
        "cluster_sizes": cluster_counts(labels),
        "core_points": int(core_mask.sum()),
        "border_points": int(border_mask.sum()),
        "noise_points": int(noise_mask.sum()),
    }
task3_fig2 = save_figure(fig, 3, 2)
task3_results["figures"]["comparison"] = task3_fig2

print(json.dumps(task3_results["eps_results"], indent=2))

{
  "0.2": {
    "n_clusters": 12,
    "cluster_sizes": {
      "noise": 188,
      "cluster_0": 293,
      "cluster_1": 33,
      "cluster_2": 15,
      "cluster_3": 12,
      "cluster_4": 8,
      "cluster_5": 12,
      "cluster_6": 8,
      "cluster_7": 5,
      "cluster_8": 11,
      "cluster_9": 5,
      "cluster_10": 4,
      "cluster_11": 6
    },
    "core_points": 362,
    "border_points": 50,
    "noise_points": 188
  },
  "0.5": {
    "n_clusters": 3,
    "cluster_sizes": {
      "noise": 21,
      "cluster_0": 300,
      "cluster_1": 267,
      "cluster_2": 12
    },
    "core_points": 565,
    "border_points": 14,
    "noise_points": 21
  },
  "1.0": {
    "n_clusters": 2,
    "cluster_sizes": {
      "noise": 2,
      "cluster_0": 300,
      "cluster_1": 298
    },
    "core_points": 594,
    "border_points": 4,
    "noise_points": 2
  }
}


![Task 2 Raw Data with bridge](figs_results/a5_task3_fig1_senthilnathan_t.png)

![Task 3 DBSCAN comparison](figs_results/a5_task3_fig2_senthilnathan_t.png)

At `eps=0.2`, DBSCAN left 188 points as noise and split the diffuse blob into many pieces. At `eps=0.5`, the left cluster was clean but the right side still had a 12-point fragment and 21 noise points. At `eps=1.0`, the method recovered two main clusters of size 300 and 298 with only 2 noise points. This shows that one global density threshold struggles when one cluster is tight and the other is much wider.

--------------------------------------------------
## Question 4: Intrinsic and Extrinsic metrics
--------------------------------------------------

In [5]:
np.random.seed(0)
X_eval, y_eval = make_blobs(n_samples=400, centers=3, cluster_std=0.7)
rows = []
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=0, n_init=20).fit_predict(X_eval)
    rows.append({
        "K": k,
        "silhouette": silhouette_score(X_eval, labels),
        "davies_bouldin": davies_bouldin_score(X_eval, labels),
        "calinski_harabasz": calinski_harabasz_score(X_eval, labels),
        "rand": rand_score(y_eval, labels),
        "fowlkes_mallows": fowlkes_mallows_score(y_eval, labels),
    })

task4_df = pd.DataFrame(rows)
fig, axes = plt.subplots(2, 3, figsize=(15.5, 9.5))
fig.subplots_adjust(hspace=0.40, wspace=0.32)
axes = axes.flatten()
metric_specs = [
    ("silhouette", "Silhouette", "higher"),
    ("davies_bouldin", "Davies-Bouldin", "lower"),
    ("calinski_harabasz", "Calinski-Harabasz", "higher"),
    ("rand", "Rand", "higher"),
    ("fowlkes_mallows", "Fowlkes-Mallows", "higher"),
]
for ax, (column, title, direction) in zip(axes, metric_specs):
    ax.plot(task4_df["K"], task4_df[column], marker="o", color="#1d3557")
    ax.set_title(title)
    ax.set_xlabel("K")
    ax.set_xticks([2, 3, 4, 5])
    ax.set_ylabel(title)
    if direction == "higher":
        best_k = int(task4_df.loc[task4_df[column].idxmax(), "K"])
    else:
        best_k = int(task4_df.loc[task4_df[column].idxmin(), "K"])
    best_value = float(task4_df.loc[task4_df["K"] == best_k, column].iloc[0])
    ax.scatter([best_k], [best_value], color="#e63946", s=40, zorder=3)
axes[-1].axis("off")
task4_fig1 = save_figure(fig, 4, 1)

task4_results = {
    "figures": {"metrics": task4_fig1},
    "metrics": task4_df.round({"silhouette": 4, "davies_bouldin": 4, "calinski_harabasz": 2, "rand": 4, "fowlkes_mallows": 4}).to_dict(orient="records"),
    "best_k": {
        "silhouette": int(task4_df.loc[task4_df["silhouette"].idxmax(), "K"]),
        "davies_bouldin": int(task4_df.loc[task4_df["davies_bouldin"].idxmin(), "K"]),
        "calinski_harabasz": int(task4_df.loc[task4_df["calinski_harabasz"].idxmax(), "K"]),
        "rand": int(task4_df.loc[task4_df["rand"].idxmax(), "K"]),
        "fowlkes_mallows": int(task4_df.loc[task4_df["fowlkes_mallows"].idxmax(), "K"]),
    }
}

print(task4_df.round(4).to_string(index=False))
print("\nBest K by metric:", task4_results["best_k"])

 K  silhouette  davies_bouldin  calinski_harabasz   rand  fowlkes_mallows
 2      0.5151          0.7250           464.0875 0.7635           0.7546
 3      0.6038          0.5464           884.7890 0.9933           0.9900
 4      0.4955          0.8716           696.8402 0.9348           0.8964
 5      0.3932          1.0309           620.3578 0.8778           0.7949

Best K by metric: {'silhouette': 3, 'davies_bouldin': 3, 'calinski_harabasz': 3, 'rand': 3, 'fowlkes_mallows': 3}


![Task 4 metric curves](figs_results/a5_task4_fig1_senthilnathan_t.png)

| K | Silhouette | Davies-Bouldin | Calinski-Harabasz | Rand | Fowlkes-Mallows |
| --- | --- | --- | --- | --- | --- |
| 2 | 0.5151 | 0.7250 | 464.09 | 0.7635 | 0.7546 |
| 3 | 0.6038 | 0.5464 | 884.79 | 0.9933 | 0.9900 |
| 4 | 0.4955 | 0.8716 | 696.84 | 0.9348 | 0.8964 |
| 5 | 0.3932 | 1.0309 | 620.36 | 0.8778 | 0.7949 |

All five metrics selected `K=3`. In this dataset that choice is clear because `K=2` merges structure and `K=4` or `K=5` starts to split clusters that are already well formed.

## Question 5: Concentric Circles

In [6]:
np.random.seed(0)
X_circle, y_true = make_circles(n_samples=600, factor=0.5, noise=0.05)
method_outputs = {
    "kmeans": KMeans(n_clusters=2, random_state=0, n_init=20).fit_predict(X_circle),
    "agglomerative_single": AgglomerativeClustering(n_clusters=2, linkage="single").fit_predict(X_circle),
    "gmm": GaussianMixture(n_components=2, random_state=0).fit(X_circle).predict(X_circle),
    "dbscan": DBSCAN(eps=0.15, min_samples=5).fit_predict(X_circle),
}

task5_rows = []
for name, labels in method_outputs.items():
    task5_rows.append({
        "method": name,
        "rand": rand_score(y_true, labels),
        "fowlkes_mallows": fowlkes_mallows_score(y_true, labels),
        "silhouette": round(safe_silhouette(X_circle, labels), 4),
        "n_clusters": int(len(set(labels)) - (1 if -1 in labels else 0)),
        "noise_points": int((labels == -1).sum()),
    })

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()
plot_clusters(axes[0], X_circle, y_true, "True labels")
plot_clusters(axes[1], X_circle, method_outputs["kmeans"], "KMeans (K=2)")
plot_clusters(axes[2], X_circle, method_outputs["agglomerative_single"], "Agglomerative single")
plot_clusters(axes[3], X_circle, method_outputs["gmm"], "Gaussian mixture (2 comps)")
plot_clusters(axes[4], X_circle, method_outputs["dbscan"], "DBSCAN eps=0.15")
axes[5].axis("off")
axes[5].text(0.0, 0.95, "Metric summary", fontsize=12, fontweight="bold", va="top")
y_pos = 0.80
for row in task5_rows:
    axes[5].text(0.0, y_pos, f"{row['method']}: Rand={row['rand']:.3f}, FM={row['fowlkes_mallows']:.3f}, Sil={row['silhouette']:.3f}", fontsize=10, va="top")
    y_pos -= 0.14
task5_fig1 = save_figure(fig, 5, 1)

task5_results = {"figures": {"comparison": task5_fig1}, "metrics": task5_rows}
print(pd.DataFrame(task5_rows).to_string(index=False))

              method     rand  fowlkes_mallows  silhouette  n_clusters  noise_points
              kmeans 0.499171         0.498336      0.3531           2             0
agglomerative_single 1.000000         1.000000      0.1131           2             0
                 gmm 0.499188         0.498630      0.3504           2             0
              dbscan 1.000000         1.000000      0.1131           2             0


![Task 5 comparison](figs_results/a5_task5_fig1_senthilnathan_t.png)

| Method | Rand | Fowlkes-Mallows | Silhouette | Clusters | Noise |
| --- | --- | --- | --- | --- | --- |
| kmeans | 0.4992 | 0.4983 | 0.3531 | 2 | 0 |
| agglomerative_single | 1.0000 | 1.0000 | 0.1131 | 2 | 0 |
| gmm | 0.4992 | 0.4986 | 0.3504 | 2 | 0 |
| dbscan | 1.0000 | 1.0000 | 0.1131 | 2 | 0 |

KMeans and GMM both produced almost the same radial split, so their Rand scores stayed near 0.50 even though their silhouette scores were much higher than the correct ring-based solutions. Agglomerative clustering with single linkage and DBSCAN recovered the two rings exactly, giving perfect Rand and Fowlkes-Mallows values. Here the external metrics reflect the true structure better than silhouette.

--------------------------------------------------
## Question 6: Spiral dataset
--------------------------------------------------

In [7]:
np.random.seed(0)
n = 300
K = 3
X_spiral = []
y_spiral = []
for k in range(K):
    theta = np.linspace(0, 4 * np.pi, n)
    r = theta
    x = r * np.cos(theta + 2 * np.pi * k / K)
    y = r * np.sin(theta + 2 * np.pi * k / K)
    data = np.column_stack([x, y])
    data += np.random.normal(scale=0.3, size=data.shape)
    X_spiral.append(data)
    y_spiral.append(np.full(n, k))
X_spiral = np.vstack(X_spiral)
y_spiral = np.concatenate(y_spiral)

eps_values = [0.3, 0.5, 0.8, 1.2]
minpts_values = [3, 5, 10]
task6_rows = []

fig, axes = plt.subplots(len(eps_values), len(minpts_values), figsize=(17, 18))
fig.subplots_adjust(hspace=0.42, wspace=0.28)
for i, eps in enumerate(eps_values):
    for j, min_samples in enumerate(minpts_values):
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_spiral)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_points = int((labels == -1).sum())
        plot_clusters(axes[i, j], X_spiral, labels, f"eps={eps}, minPts={min_samples}\nclusters={n_clusters}, noise={noise_points}")
        task6_rows.append({
            "eps": eps,
            "min_samples": min_samples,
            "n_clusters": int(n_clusters),
            "noise_points": noise_points,
            "rand": round(rand_score(y_spiral, labels), 4),
            "fowlkes_mallows": round(fowlkes_mallows_score(y_spiral, labels), 4),
        })
task6_fig1 = save_figure(fig, 6, 1)

best_dbscan = max(task6_rows, key=lambda row: row["rand"])
best_labels = DBSCAN(eps=best_dbscan["eps"], min_samples=best_dbscan["min_samples"]).fit_predict(X_spiral)
kmeans_labels = KMeans(n_clusters=3, random_state=0, n_init=20).fit_predict(X_spiral)
kmeans_scores = {
    "rand": round(rand_score(y_spiral, kmeans_labels), 4),
    "fowlkes_mallows": round(fowlkes_mallows_score(y_spiral, kmeans_labels), 4),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
plot_clusters(axes[0], X_spiral, y_spiral, "True spiral labels")
plot_clusters(axes[1], X_spiral, kmeans_labels, f"KMeans (K=3)\nRand={kmeans_scores['rand']:.3f}, FM={kmeans_scores['fowlkes_mallows']:.3f}")
plot_clusters(axes[2], X_spiral, best_labels, f"Best DBSCAN in grid\neps={best_dbscan['eps']}, minPts={best_dbscan['min_samples']}, Rand={best_dbscan['rand']:.3f}")
task6_fig2 = save_figure(fig, 6, 2)

task6_results = {
    "figures": {"dbscan_grid": task6_fig1, "comparison": task6_fig2},
    "dbscan_results": task6_rows,
    "best_dbscan": best_dbscan,
    "kmeans_scores": kmeans_scores,
}

print(pd.DataFrame(task6_rows).to_string(index=False))
print("\nBest DBSCAN:", best_dbscan)
print("KMeans baseline:", kmeans_scores)

 eps  min_samples  n_clusters  noise_points   rand  fowlkes_mallows
 0.3            3          53           585 0.5226           0.3834
 0.3            5           6           764 0.4232           0.4938
 0.3           10           1           847 0.3696           0.5436
 0.5            3          73           280 0.6205           0.2422
 0.5            5          24           539 0.5338           0.3721
 0.5           10           2           766 0.4184           0.4971
 0.8            3          64            26 0.6007           0.3083
 0.8            5          30           239 0.5851           0.3063
 0.8           10           6           594 0.5017           0.4106
 1.2            3           7             0 0.3999           0.5212
 1.2            5          17             9 0.4326           0.4881
 1.2           10           9           386 0.5244           0.3782

Best DBSCAN: {'eps': 0.5, 'min_samples': 3, 'n_clusters': 73, 'noise_points': 280, 'rand': 0.6205, 'fowlkes_mallows

The taskfile explicitly asks for DBSCAN and then refers to both methods. To keep the comparison meaningful, I used KMeans as the distance-based baseline.

![Task 6 DBSCAN grid](figs_results/a5_task6_fig1_senthilnathan_t.png)

![Task 6 comparison](figs_results/a5_task6_fig2_senthilnathan_t.png)

| eps | minPts | Clusters | Noise | Rand | Fowlkes-Mallows |
| --- | --- | --- | --- | --- | --- |
| 0.3 | 3 | 53 | 585 | 0.5226 | 0.3834 |
| 0.3 | 5 | 6 | 764 | 0.4232 | 0.4938 |
| 0.3 | 10 | 1 | 847 | 0.3696 | 0.5436 |
| 0.5 | 3 | 73 | 280 | 0.6205 | 0.2422 |
| 0.5 | 5 | 24 | 539 | 0.5338 | 0.3721 |
| 0.5 | 10 | 2 | 766 | 0.4184 | 0.4971 |
| 0.8 | 3 | 64 | 26 | 0.6007 | 0.3083 |
| 0.8 | 5 | 30 | 239 | 0.5851 | 0.3063 |
| 0.8 | 10 | 6 | 594 | 0.5017 | 0.4106 |
| 1.2 | 3 | 7 | 0 | 0.3999 | 0.5212 |
| 1.2 | 5 | 17 | 9 | 0.4326 | 0.4881 |
| 1.2 | 10 | 9 | 386 | 0.5244 | 0.3782 |

Inside the required parameter grid, the highest Rand score was 0.6205 at `eps=0.5` and `minPts=3`. Even then DBSCAN created 73 clusters and 280 noise points, so it only followed short local segments. KMeans gave Rand=0.5510, which is lower, but its partition still cuts across the curved arms. The spiral shape is hard because the local spacing changes from the center to the outer region.

## Final Summary

Across these tasks, the main lesson is that clustering quality depends on the shape and density pattern of the data. When the groups are simple and separated, several methods agree. When connectivity, varying density, or curved geometry enters the picture, the method assumptions become much more visible in the final result.